In [33]:
# Install Packages
!pip install google-api-python-client pycountry isodate

In [ ]:
# --- 1. SETUP AND AUTHENTICATION ---

from googleapiclient.discovery import build
from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from datetime import datetime, timedelta
from datetime import datetime
import pandas as pd
import pycountry
import isodate
import time
import os

# Replace this with your actual API key
API_KEY = os.getenv("YOUTUBE_API_KEY1"),

# Initialize the YouTube Data API client
youtube = build("youtube", "v3", developerKey=API_KEY)

print("✅ YouTube client initialized successfully")

✅ YouTube client initialized successfully


In [3]:
# Initialize BigQuery client
client = bigquery.Client(project='data-storage-485106')

In [4]:
now = datetime.now()
year = now.year
month = now.strftime("%b").lower()  # jan, feb, mar

table_suffix = f"{year}_{month}"
table_suffix

'2026_may'

In [24]:
sql = """
    SELECT DISTINCT query, MAX(search_volume) AS search_volume
    FROM `data-storage-485106.google.stg_trending_now`
    WHERE start_date >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
    GROUP BY query
"""

trends_df = client.query(sql).to_dataframe()
trending_keywords = trends_df["query"].tolist()
print(f"Keywords from last 24 hours: {len(trending_keywords)}")

Keywords from last 24 hours: 32


In [25]:
trending_keywords

['aston villa vs liverpool',
 'youtube videos',
 'iceman drake',
 'inua jamii biometric verification',
 'valencia vs rayo vallecano prediction',
 'sony xperia 1 viii',
 'eberechi eze arsenal transfer bonuses',
 'valencia fc',
 'power',
 'the university of melbourne',
 'gasoline',
 'michael carrick',
 'epra',
 'mikel arteta',
 'bradford city vs bolton',
 'girona',
 'valencia',
 'union saint-gilloise vs anderlecht',
 'real madrid fc',
 'breaking news',
 'teachers service commission',
 'copenhagen vs midtjylland',
 'diesel fuel',
 'al-ettifaq vs al-ittihad',
 'pbks vs mi',
 'real madrid',
 'benjamin netanyahu',
 'epra kenya',
 'fa youth cup final',
 'real madrid vs real oviedo',
 'epra fuel prices',
 'emurua dikirr by election results']

#### **Youtube Trending**

In [ ]:
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

In [ ]:
REGION    = "KE"
MAX_PAGES = 2

API_KEYS = [
    os.getenv("YOUTUBE_API_KEY1"),
    os.getenv("YOUTUBE_API_KEY2"),
    os.getenv("YOUTUBE_API_KEY3"),
    os.getenv("YOUTUBE_API_KEY4"),
]

current_key_index = 0

def get_youtube_client(index):
    return build('youtube', 'v3', developerKey=API_KEYS[index])

youtube = get_youtube_client(current_key_index)

seven_days_ago = (datetime.utcnow() - timedelta(days=7)).strftime('%Y-%m-%dT%H:%M:%SZ')

# Build category map once
cat_response = youtube.videoCategories().list(part="snippet", regionCode=REGION).execute()
category_map = {item["id"]: item["snippet"]["title"] for item in cat_response.get("items", [])}

results = []

In [28]:
for keyword in trending_keywords:
    print(f"🔍 Searching: '{keyword}'")
    video_ids = []
    next_page_token = None

    for page in range(MAX_PAGES):
        try:
            search_response = youtube.search().list(
                part="id",
                q=keyword,
                type="video",
                regionCode=REGION,
                maxResults=50,
                order="date",
                publishedAfter=seven_days_ago,
                pageToken=next_page_token
            ).execute()
        except HttpError as e:
            if "quotaExceeded" in str(e):
                current_key_index += 1
                if current_key_index >= len(API_KEYS):
                    print("❌ All API keys exhausted, stopping.")
                    break
                print(f"⚠️ Quota exceeded, switching to key {current_key_index + 1}...")
                youtube = get_youtube_client(current_key_index)
                search_response = youtube.search().list(
                    part="id",
                    q=keyword,
                    type="video",
                    regionCode=REGION,
                    maxResults=50,
                    order="date",
                    publishedAfter=seven_days_ago,
                    pageToken=next_page_token
                ).execute()
            else:
                raise

        for item in search_response.get("items", []):
            vid_id = item.get("id", {}).get("videoId")
            if vid_id:
                video_ids.append(vid_id)

        next_page_token = search_response.get("nextPageToken")
        if not next_page_token:
            break

    # Fetch full stats for all video IDs collected for this keyword
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i + 50]
        try:
            response = youtube.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(chunk)
            ).execute()
        except HttpError as e:
            if "quotaExceeded" in str(e):
                current_key_index += 1
                if current_key_index >= len(API_KEYS):
                    print("❌ All API keys exhausted during stats fetch, stopping.")
                    break
                print(f"⚠️ Quota exceeded during stats fetch, switching to key {current_key_index + 1}...")
                youtube = get_youtube_client(current_key_index)
                response = youtube.videos().list(
                    part="snippet,statistics,contentDetails",
                    id=",".join(chunk)
                ).execute()
            else:
                raise

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            stats   = item.get("statistics", {})
            cat_id  = snippet.get("categoryId", "")
            results.append({
                "keyword":       keyword,
                "video_id":      item.get("id"),
                "title":         snippet.get("title"),
                "channel_title": snippet.get("channelTitle"),
                "category_name": category_map.get(cat_id, "Unknown"),
                "published_at":  snippet.get("publishedAt"),
                "duration":      item.get("contentDetails", {}).get("duration"),
                "tags":          str(snippet.get("tags", [])),
                "view_count":    int(stats.get("viewCount",    0)),
                "like_count":    int(stats.get("likeCount",    0)),
                "comment_count": int(stats.get("commentCount", 0)),
                "region_code":   REGION,
            })

    time.sleep(0.5)

data = pd.DataFrame(results)
print(f"\n✅ Done — shape: {data.shape}")

🔍 Searching: 'aston villa vs liverpool'
🔍 Searching: 'youtube videos'
🔍 Searching: 'iceman drake'
🔍 Searching: 'inua jamii biometric verification'
🔍 Searching: 'valencia vs rayo vallecano prediction'
🔍 Searching: 'sony xperia 1 viii'
🔍 Searching: 'eberechi eze arsenal transfer bonuses'
🔍 Searching: 'valencia fc'
🔍 Searching: 'power'
🔍 Searching: 'the university of melbourne'
🔍 Searching: 'gasoline'
🔍 Searching: 'michael carrick'
🔍 Searching: 'epra'
🔍 Searching: 'mikel arteta'
🔍 Searching: 'bradford city vs bolton'
🔍 Searching: 'girona'
🔍 Searching: 'valencia'
🔍 Searching: 'union saint-gilloise vs anderlecht'
🔍 Searching: 'real madrid fc'
🔍 Searching: 'breaking news'
🔍 Searching: 'teachers service commission'
🔍 Searching: 'copenhagen vs midtjylland'
🔍 Searching: 'diesel fuel'
🔍 Searching: 'al-ettifaq vs al-ittihad'
🔍 Searching: 'pbks vs mi'
🔍 Searching: 'real madrid'
🔍 Searching: 'benjamin netanyahu'
🔍 Searching: 'epra kenya'
🔍 Searching: 'fa youth cup final'
🔍 Searching: 'real madrid v

In [45]:
import isodate

def duration_to_seconds(d):
    try:
        return int(isodate.parse_duration(d).total_seconds())
    except:
        return 0

data["duration_secs"] = data["duration"].apply(duration_to_seconds)

bigdata = data[
    (data["duration_secs"] >= 60)   # minimum 1 minute
    & (data["view_count"] > 0)      # at least 1 view
].copy()

print(f"Before cleaning: {data.shape[0]} rows")
print(f"After cleaning:  {bigdata.shape[0]} rows")

Before cleaning: 2430 rows
After cleaning:  1252 rows


In [46]:
bigdata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1252 entries, 7 to 2429
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   keyword        1252 non-null   object
 1   video_id       1252 non-null   object
 2   title          1252 non-null   object
 3   channel_title  1252 non-null   object
 4   category_name  1252 non-null   object
 5   published_at   1252 non-null   object
 6   duration       1252 non-null   object
 7   tags           1252 non-null   object
 8   view_count     1252 non-null   int64 
 9   like_count     1252 non-null   int64 
 10  comment_count  1252 non-null   int64 
 11  region_code    1252 non-null   object
 12  duration_secs  1252 non-null   int64 
dtypes: int64(4), object(9)
memory usage: 136.9+ KB


In [47]:
bigdata

,keyword,video_id,title,channel_title,category_name,published_at,duration,tags,view_count,like_count,comment_count,region_code,duration_secs
7,aston villa vs liverpool,QYcNsAqPsI0,Liverpool vs Villa 3 best bets,MP Picks,People & Blogs,2026-05-15T05:56:54Z,PT1M3S,[],178,1,0,KE,63
8,aston villa vs liverpool,-xJpEUMgZIE,EA FC 26 - Aston Villa vs Liverpool | Premier ...,Feraz Sports,Gaming,2026-05-15T05:51:00Z,PT17M4S,"['PS5Share', 'ShareFactoryStudio']",2,1,0,KE,1024
11,aston villa vs liverpool,nK90WIhhepg,Trực tiếp nhận định Aston Villa vs Liverpool -...,Dy vlog,People & Blogs,2026-05-15T05:15:06Z,PT3M6S,[],4,0,0,KE,186
14,aston villa vs liverpool,gC87dlHqQhY,Aston Villa vs Liverpool - Euro Football Today...,Theodor Valentin,Sports,2026-05-15T05:00:09Z,PT2M26S,"['fotbal', 'stiri fotbal', 'euro football toda...",1,0,0,KE,146
15,aston villa vs liverpool,O23OKbeOtS0,Aston Villa vs Liverpool - Euro Football Today...,Euro Football Today,Sports,2026-05-15T05:00:23Z,PT2M11S,"['Euro Football Today', 'football shorts', 'Pr...",3,1,0,KE,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2425,emurua dikirr by election results,tT65rysi7uY,Ol Kalou & Emurua Dikirr by-elections: DCP vs ...,TV47 Kenya,News & Politics,2026-05-12T05:50:45Z,PT21M6S,"['News', 'Kenya News', 'TV47', 'Live News', 'B...",5498,19,19,KE,1266
2426,emurua dikirr by election results,j_O-J2tnML8,EMURUA DIKIRR RESULTS COULD DESTROY DCP MOMENTUM!,The Republic Watch,News & Politics,2026-05-12T05:35:00Z,PT8M5S,[],4203,33,18,KE,485
2427,emurua dikirr by election results,H3-d7uBQpwg,LIVE- Breaking David Keter UDA Emurua Dikirr h...,Kenya Digital News,News & Politics,2026-05-12T03:38:18Z,PT59M6S,[],2756,6,0,KE,3546
2428,emurua dikirr by election results,39K_ULbQj1g,Will DCP or UDA win in the Ol Kalou and Emurua...,TV47 Kenya,News & Politics,2026-05-11T05:42:39Z,PT8M21S,"['News', 'Kenya News', 'TV47', 'Live News', 'B...",8971,33,15,KE,501


In [ ]:
# Define Table ID
table_id = f"data-storage-485106.youtube.trending_l24h_{table_suffix}"

prev_month_date = now.replace(day=1) - timedelta(days=1)
prev_table_suffix = f"{prev_month_date.year}_{prev_month_date.strftime('%b').lower()}"
prev_table_id = f"data-storage-485106.youtube.trending_l24h_{prev_table_suffix}"

if now.day <= 7:

    # Count how many prev-month rows exist in the SOURCE (prev) table
    try:
        prev_count_df = client.query(
            f"SELECT COUNT(*) AS cnt FROM `{prev_table_id}`"
        ).to_dataframe()
        prev_table_total = prev_count_df.loc[0, "cnt"]
    except NotFound:
        prev_table_total = 0  # No previous table at all

    # Count how many prev-month rows already exist in the NEW table
    try:
        check_sql = f"""
            SELECT COUNT(*) AS cnt
            FROM `{table_id}`
            WHERE EXTRACT(MONTH FROM TIMESTAMP(published_at)) = {prev_month_date.month}
            AND EXTRACT(YEAR FROM TIMESTAMP(published_at)) = {prev_month_date.year}
        """
        check_df = client.query(check_sql).to_dataframe()
        existing_prev_in_new = check_df.loc[0, "cnt"]
    except NotFound:
        existing_prev_in_new = 0  # Table doesn't exist yet

    # Only treat it as "fully merged" if counts match (or there's nothing to merge)
    fully_merged = (prev_table_total == 0) or (existing_prev_in_new >= prev_table_total)

    if not fully_merged:
        # ROLLOVER WORKFLOW - merge, deduping against rows already leaked in
        try:
            try:
                prev_data = client.query(
                    f"SELECT * FROM `{prev_table_id}` ORDER BY published_at DESC"
                ).to_dataframe()

                if existing_prev_in_new > 0:
                    # Some June rows already leaked into table_id - dedupe before appending
                    existing = client.query(
                        f"""
                        SELECT video_id, published_at FROM `{table_id}`
                        WHERE EXTRACT(MONTH FROM TIMESTAMP(published_at)) = {prev_month_date.month}
                        AND EXTRACT(YEAR FROM TIMESTAMP(published_at)) = {prev_month_date.year}
                        """
                    ).to_dataframe()
                    prev_data = prev_data.merge(
                        existing, on=["video_id", "published_at"], how="left", indicator=True
                    )
                    prev_data = prev_data[prev_data["_merge"] == "left_only"].drop(columns=["_merge"])
                    print(f"Deduped: {len(prev_data)} new previous-month rows remain to append.")

                bigdata = pd.concat([prev_data, bigdata], ignore_index=True)
                print(f"Appended {len(prev_data)} rows from previous month table ({prev_table_id}).")
            except NotFound:
                print(f"No previous month table found ({prev_table_id}), skipping append.")

            job = client.load_table_from_dataframe(
                bigdata,
                table_id,
                job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
            )
            job.result()
            print(f"Rollover load completed into {table_id}, total rows: {len(bigdata)}")

        except Exception as e:
            print(f"Error during month-rollover load: {e}")

    else:
        # Already fully merged, just do normal load
        job = client.load_table_from_dataframe(
            bigdata,
            table_id,
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
        )
        job.result()
        print(f"Normal load completed into {table_id}, rows: {len(bigdata)}")

else:
    # NORMAL WORKFLOW (outside the 7-day window)
    job = client.load_table_from_dataframe(
        bigdata,
        table_id,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    )
    job.result()
    print(f"Normal load completed into {table_id}, rows: {len(bigdata)}")

In [ ]:
# # Define Table ID
# table_id = f"data-storage-485106.youtube.trending_l24h_{table_suffix}"

# if now.day == 1 or now.day == 2 or now.day == 5:

#     # Check if current month table already has current month data
#     try:
#         check_sql = f"""
#                             SELECT COUNT(*) AS cnt
#                             FROM `{table_id}`
#                             WHERE EXTRACT(MONTH FROM TIMESTAMP(published_at)) = {now.month}
#                             AND EXTRACT(YEAR FROM TIMESTAMP(published_at)) = {now.year}
#                             """
#         check_df = client.query(check_sql).to_dataframe()
#         has_current_month_data = check_df.loc[0, "cnt"] > 0
#     except NotFound:
#         has_current_month_data = False  # Table doesn't exist yet
  
#     if not has_current_month_data:
#       try:
#         prev_month_date = now.replace(day=1) - timedelta(days=1)
#         prev_table_suffix = f"{prev_month_date.year}_{prev_month_date.strftime('%b').lower()}"
#         prev_table_id = f"data-storage-485106.youtube.trending_l24h_{prev_table_suffix}"
        
#         try:
#             prev_data = client.query(
#                 f"SELECT * FROM `{prev_table_id}` ORDER BY published_at DESC"
#             ).to_dataframe()
#             bigdata = pd.concat([prev_data, bigdata], ignore_index=True)
#             print(f"Appended {len(prev_data)} rows from previous month table.")
#         except NotFound:
#             print("No previous month table found, skipping append.")
        
#         job = client.load_table_from_dataframe(
#             bigdata,
#             table_id,
#             job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
#         )
#         job.result()
#         print(f"All data loaded into {table_id}, total rows: {len(bigdata)}")

#       except Exception as e:
#           print(f"Error during 1st-of-month load: {e}")

# else:
#     # 🔥 NORMAL WORKFLOW (this was missing)
#     job = client.load_table_from_dataframe(
#         bigdata,
#         table_id,
#         job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
#     )
#     job.result()
#     print(f"Normal load completed into {table_id}, rows: {len(bigdata)}")

Normal load completed into data-storage-485106.youtube.trending_l7d_2026_may, rows: 1252


In [50]:
# Define Table ID
table_id = f"data-storage-485106.youtube.trending_l24h_{table_suffix}"

# Define SQL Query to Retrieve Open Weather Data from Google Cloud BigQuery
sql = (f"""
        SELECT *
        FROM `{table_id}`
       """)
    
# Run SQL Query
data = client.query(sql).to_dataframe()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [51]:
data.head()

,keyword,video_id,title,channel_title,category_name,published_at,duration,tags,view_count,like_count,comment_count,region_code,duration_secs
0,gasoline,Wjpmj_QEIhg,Big Costs And Questionable Quality - Gasoline ...,Uncle Tony's Garage,Autos & Vehicles,2026-05-14T02:20:05Z,PT2H11M40S,[],7481,385,49,KE,7900
1,youtube videos,FgVMxsIh8f4,Why I Stopped Filming YouTube Videos in 2026,Julia McCoy,Education,2026-05-14T15:00:07Z,PT10M57S,[],2511,215,28,KE,657
2,youtube videos,8MCXyMbw3Dc,USA Stick Animation Video Kaise Banaye 🔥 USA F...,Invisible Gyan,Education,2026-05-13T13:50:00Z,PT18M27S,"['Invisible Gyan', 'youtube channel ideas', 'f...",16947,0,188,KE,1107
3,power,6jYrTjV3Jdw,ADANI GROUP अब तैयार रहना 🔥 ADANI POWER LATEST...,Stock Market Adviser,Education,2026-05-14T13:18:22Z,PT13M14S,"['adani group news', 'adani group latest news'...",4427,160,37,KE,794
4,breaking news,M4vLAMxBZOg,Breaking News 🔥 Petrol Diesel Price Hike #test...,Testbook,Education,2026-05-15T03:18:15Z,PT1M15S,[],5660,577,55,KE,75


In [ ]:
# Define Table ID
table_id = f"data-storage-485106.youtube.trending_l24h_{table_suffix}"

# Delete Original Table
client.delete_table(table_id)

# Remove Duplicate Records
data.drop_duplicates(subset=["keyword", "video_id", "region_code"], inplace=True)
print(f"Table deleted successfully.")

In [56]:
if 'tags' in bigdata.columns:
    bigdata['tags'] = bigdata['tags'].astype(str)
else:
    bigdata['tags'] = None

In [57]:
# Define the dataset ID and table ID
dataset_id = 'youtube'
table_id = f"trending_l24h_{table_suffix}"
    
# Define the BigQuery schema for YouTube trending videos
schema = [
    bigquery.SchemaField("keyword",       "STRING"),
    bigquery.SchemaField("video_id",      "STRING"),
    bigquery.SchemaField("title",         "STRING"),
    bigquery.SchemaField("channel_title", "STRING"),
    bigquery.SchemaField("category_name", "STRING"),
    bigquery.SchemaField("published_at",  "STRING"),
    bigquery.SchemaField("duration",      "STRING"),
    bigquery.SchemaField("duration_secs", "INTEGER"),
    bigquery.SchemaField("tags",          "STRING"),
    bigquery.SchemaField("view_count",    "INTEGER"),
    bigquery.SchemaField("like_count",    "INTEGER"),
    bigquery.SchemaField("comment_count", "INTEGER"),
    bigquery.SchemaField("region_code",   "STRING"),
]

# Define the table reference
table_ref = client.dataset(dataset_id).table(table_id)
    
# Create the table object
table = bigquery.Table(table_ref, schema=schema)

try:
    # Create the table in BigQuery
    table = client.create_table(table)
    print(f"Table {table.table_id} created successfully.")
except Exception as e:
    print(f"Table {table.table_id} failed")

Table trending_l7d_2026_may created successfully.


In [59]:
# Define Table ID
table_id = f"data-storage-485106.youtube.trending_l24h_{table_suffix}"

# Load the data into the BigQuery table
job = client.load_table_from_dataframe(data, table_id)

# Wait for the job to complete
while job.state != 'DONE':
    time.sleep(2)
    job.reload()
    print(job.state)

DONE


In [60]:
# Define Table ID
table_id = f"data-storage-485106.youtube.trending_l24h_{table_suffix}"

# Define SQL Query to Retrieve Open Weather Data from Google Cloud BigQuery
sql = (f"""
        SELECT *
        FROM `{table_id}`
       """)
    
# Run SQL Query
data = client.query(sql).to_dataframe()
print(f'Shape : {data.shape}')

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Shape : (1245, 13)


In [61]:
data

,keyword,video_id,title,channel_title,category_name,published_at,duration,duration_secs,tags,view_count,like_count,comment_count,region_code
0,gasoline,Wjpmj_QEIhg,Big Costs And Questionable Quality - Gasoline ...,Uncle Tony's Garage,Autos & Vehicles,2026-05-14T02:20:05Z,PT2H11M40S,7900,[],7481,385,49,KE
1,youtube videos,FgVMxsIh8f4,Why I Stopped Filming YouTube Videos in 2026,Julia McCoy,Education,2026-05-14T15:00:07Z,PT10M57S,657,[],2511,215,28,KE
2,youtube videos,8MCXyMbw3Dc,USA Stick Animation Video Kaise Banaye 🔥 USA F...,Invisible Gyan,Education,2026-05-13T13:50:00Z,PT18M27S,1107,"['Invisible Gyan', 'youtube channel ideas', 'f...",16947,0,188,KE
3,power,6jYrTjV3Jdw,ADANI GROUP अब तैयार रहना 🔥 ADANI POWER LATEST...,Stock Market Adviser,Education,2026-05-14T13:18:22Z,PT13M14S,794,"['adani group news', 'adani group latest news'...",4427,160,37,KE
4,breaking news,M4vLAMxBZOg,Breaking News 🔥 Petrol Diesel Price Hike #test...,Testbook,Education,2026-05-15T03:18:15Z,PT1M15S,75,[],5660,577,55,KE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1240,bradford city vs bolton,poHX-HZwmDI,BOLTON BOOK WEMBLEY! | Bradford City v Bolton ...,EFL,Sports,2026-05-14T23:06:49Z,PT12M54S,774,"['EFL', 'Goal', 'Save', 'Championship', 'Leagu...",37090,574,46,KE
1241,real madrid fc,syWjUXU27M0,La limpia de 9 jugadores del Real Madrid,Adictos Futbol,Sports,2026-05-13T23:01:01Z,PT1M20S,80,"['La limpia de 9 jugadores del Real Madrid', '...",297905,3425,46,KE
1242,real madrid,syWjUXU27M0,La limpia de 9 jugadores del Real Madrid,Adictos Futbol,Sports,2026-05-13T23:01:01Z,PT1M20S,80,"['La limpia de 9 jugadores del Real Madrid', '...",297905,3426,46,KE
1243,real madrid,qxJKtXUSw7s,"""We Know What Happened Before!"" Ramon Calderon...",talkSPORT,Sports,2026-05-13T15:01:30Z,PT8M42S,522,"['Arsenal', 'Chelsea', 'H&J', 'Hawksbee & Jaco...",9964,73,46,KE
